# RosenblattVinedist demo

This small example fits a conditional distribution on the original data scale. `RosenblattVinedist` combines conditional margins with a fixed-structure `RosenblattVinecop`.

We compare three configurations: `tabpfn-criterion`, `ngboost`, and `gbm`. Each configuration uses one shared backend family for its margins and pair copulas.

## Setup

Install one PyTorch flavor and the optional classical backends, for example `uv sync --extra cu130 --extra ngboost --extra gbm --extra interactive`. TabPFN may require credentials and an initial model download. A failed or unavailable backend is reported and skipped.

In [ ]:
from time import perf_counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyvinecopulib as pv
import torch
from dotenv import load_dotenv
from scipy.stats import kstest

from npcc import QuantileTableConfig, RosenblattVinedist, create_backend
from npcc.core.controls import FitControlsRosenblattVinecop
from npcc.core.vinecop import RosenblattVinecop

load_dotenv()

SEED = 42
N_TRAIN = 250
N_TEST = 100
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
rng = np.random.default_rng(SEED)

print(f"device: {DEVICE}")

## Conditional data on the original scale

A scalar covariate changes both the marginal means and the dependence. The three conditional margins are Gaussian with different location and scale functions. Their latent normal variables share a common factor whose strength increases with the covariate.

In [ ]:
def conditional_means(x: np.ndarray) -> np.ndarray:
  values = np.asarray(x, dtype=np.float64).reshape(-1)
  return np.column_stack(
    [
      1.0 + 0.8 * values,
      -0.5 + 0.4 * values,
      2.0 - 0.6 * values,
    ]
  )


def generate_data(
  n: int, generator: np.random.Generator
) -> tuple[np.ndarray, np.ndarray]:
  x = generator.uniform(-1.0, 1.0, size=(n, 1))
  rho = 0.15 + 0.65 * (x + 1.0) / 2.0
  common = generator.standard_normal((n, 1))
  noise = generator.standard_normal((n, 3))
  z = np.sqrt(rho) * common + np.sqrt(1.0 - rho) * noise
  scale = np.column_stack(
    [
      0.55 + 0.10 * (x[:, 0] + 1.0),
      0.75 + 0.10 * (x[:, 0] + 1.0),
      0.45 + 0.15 * (x[:, 0] + 1.0),
    ]
  )
  return conditional_means(x) + scale * z, x


y_train, x_train = generate_data(N_TRAIN, rng)
y_test, x_test = generate_data(N_TEST, rng)

fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
for j, ax in enumerate(axes):
  ax.scatter(x_train[:, 0], y_train[:, j], s=12, alpha=0.55)
  order = np.argsort(x_train[:, 0])
  truth = conditional_means(x_train)[:, j]
  ax.plot(x_train[order, 0], truth[order], color="black", linewidth=2)
  ax.set(xlabel="x", ylabel=f"y{j + 1}", title=f"Margin {j + 1}")
fig.tight_layout()
plt.show()

## Fit three backend configurations

All models use the same three-dimensional D-vine structure and quantile grid. The small backend-specific settings keep this a demonstration rather than a tuning exercise.

In [ ]:
structure = pv.RVineStructure.from_order([1, 2, 3])
quantile_table_config = QuantileTableConfig(n_quantiles=41)
backend_kwargs = {
  "tabpfn-criterion": {},
  "ngboost": {"n_estimators": 150, "learning_rate": 0.04},
  "gbm": {"n_estimators": 80, "max_depth": 3},
}


def synchronize_device() -> None:
  if DEVICE.type == "cuda":
    torch.cuda.synchronize(DEVICE)


models: dict[str, RosenblattVinedist] = {}
fit_records: list[dict[str, object]] = []
for backend, kwargs in backend_kwargs.items():
  synchronize_device()
  started = perf_counter()
  try:
    controls = FitControlsRosenblattVinecop(
      backend=backend,
      quantile_table_config=quantile_table_config,
      device=DEVICE,
      backend_kwargs=kwargs,
    )
    margins = [
      create_backend(
        backend,
        transform="identity",
        quantile_table_config=quantile_table_config,
        eps=controls.eps,
        device=DEVICE,
        batch_size=controls.batch_size,
        backend_kwargs=kwargs,
      )
      for _ in range(structure.dim)
    ]
    vinecop = RosenblattVinecop(None, structure, device=DEVICE)
    model = RosenblattVinedist(vinecop, margins).fit(
      y_train, controls, x=x_train
    )
    synchronize_device()
    elapsed = perf_counter() - started
    models[backend] = model
    held_out = model.logpdf(y_test, x=x_test).detach().cpu().numpy()
    fit_records.append(
      {
        "backend": backend,
        "status": "ok",
        "seconds": elapsed,
        "mean_test_logpdf": float(np.mean(held_out)),
        "error": "",
      }
    )
  except (ImportError, RuntimeError) as exc:
    synchronize_device()
    fit_records.append(
      {
        "backend": backend,
        "status": "failed",
        "seconds": perf_counter() - started,
        "mean_test_logpdf": np.nan,
        "error": f"{type(exc).__name__}: {exc}",
      }
    )

fit_report = pd.DataFrame(fit_records)
display(fit_report)

if not models:
  raise RuntimeError("No backend could be fitted; inspect the report.")

## Conditional marginal calibration

For a calibrated conditional margin, $F_j(Y_j\mid X)$ is uniform. The table reports a Kolmogorov–Smirnov statistic for each fitted margin, and the histograms pool the three transformed columns for a compact visual check.

In [ ]:
pit_by_backend = {
  backend: model.marginal_cdf(y_test, x=x_test).detach().cpu().numpy()
  for backend, model in models.items()
}
pit_report = pd.DataFrame(
  [
    {
      "backend": backend,
      "margin": j + 1,
      "ks_statistic": float(kstest(pit[:, j], "uniform").statistic),
      "mean": float(np.mean(pit[:, j])),
      "std": float(np.std(pit[:, j])),
    }
    for backend, pit in pit_by_backend.items()
    for j in range(3)
  ]
)
display(pit_report)

fig, axes = plt.subplots(1, len(models), figsize=(4 * len(models), 3.5))
axes = np.atleast_1d(axes)
for ax, (backend, pit) in zip(axes, pit_by_backend.items()):
  ax.hist(pit.ravel(), bins=10, range=(0, 1), density=True, alpha=0.75)
  ax.axhline(1.0, color="black", linestyle="--")
  ax.set(xlabel="conditional PIT", ylabel="density", title=backend)
fig.tight_layout()
plt.show()

## Samples on the original scale

We draw repeated observations at three covariate values. The estimated conditional sample means are compared with the known marginal mean functions.

In [ ]:
x_levels = np.array([-0.75, 0.0, 0.75])
draws_per_level = 100
x_sample = np.repeat(x_levels, draws_per_level)[:, None]
sample_records: list[dict[str, object]] = []

for backend, model in models.items():
  samples = (
    model.sample(len(x_sample), x=x_sample, seeds=[SEED]).detach().cpu().numpy()
  )
  for level_index, level in enumerate(x_levels):
    rows = slice(
      level_index * draws_per_level,
      (level_index + 1) * draws_per_level,
    )
    for j in range(3):
      sample_records.append(
        {
          "backend": backend,
          "x": level,
          "margin": j + 1,
          "sample_mean": float(np.mean(samples[rows, j])),
          "true_mean": float(conditional_means(np.array([[level]]))[0, j]),
        }
      )

sample_report = pd.DataFrame(sample_records)
display(sample_report)

fig, axes = plt.subplots(1, 3, figsize=(13, 3.5), sharex=True)
for j, ax in enumerate(axes, start=1):
  subset = sample_report[sample_report["margin"] == j]
  for backend in models:
    rows = subset[subset["backend"] == backend]
    ax.plot(rows["x"], rows["sample_mean"], marker="o", label=backend)
  truth = subset.drop_duplicates("x").sort_values("x")
  ax.plot(truth["x"], truth["true_mean"], "k--", label="truth")
  ax.set(xlabel="x", ylabel="conditional mean", title=f"Margin {j}")
axes[0].legend(fontsize=8)
fig.tight_layout()
plt.show()

## Takeaways

- Each response column owns an independently fitted backend margin.
- The conditional marginal CDFs provide the pseudo-observations used by the vine.
- Margin and pair-copula backends can be selected independently even though this demo pairs like with like.
- This small sample is meant to demonstrate the API, not rank the backends. Use larger held-out datasets and repeated fits for comparisons.